# TFM Tenerife — BERTopic MODELO B ("por ubicación") — Booking + TripAdvisor + LosViajeros georreferenciado

Corre esto en Colab con GPU (**Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**, gratis).

38.341 documentos (tras deduplicar texto exacto): 40.029 Booking + 807 TripAdvisor + 990 LosViajeros con ubicación detectada. En CPU local esto no es viable en un tiempo razonable — por eso se entrena aquí.

**Pasos**: 1) ejecuta todas las celdas en orden (Entorno de ejecución → Ejecutar todo). 2) Cuando te lo pida, sube `geo_corpus.csv`. 3) Al final se descargan dos archivos: `geo_topics_results.csv` (para cargar en Azure) y `bertopic_model_geo.zip` (el modelo entrenado, opcional). 4) Pásaselos a Claude para que los suba a `gold.nlp_topics`.

Misma configuración validada en el Modelo A (ver `analytics/contexto.md` del repo para el porqué de cada decisión: 5 iteraciones para llegar a esto).

In [ ]:
!pip install -q bertopic sentence-transformers

import torch
print('GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('AVISO: sin GPU. Ve a Entorno de ejecucion > Cambiar tipo de entorno de ejecucion > T4 GPU, y vuelve a ejecutar.')

In [ ]:
from google.colab import files
print('Sube geo_corpus.csv (generado por analytics/topics/export_geo_corpus.py):')
uploaded = files.upload()

In [ ]:
import pandas as pd

nombre_csv = list(uploaded.keys())[0]
df = pd.read_csv(nombre_csv)
print(f'{len(df)} documentos cargados.')
print(df['source'].value_counts())

textos = df['text'].tolist()

## Configuración del modelo

Misma que el Modelo A (YouTube), validada tras varias pruebas -- ver `analytics/contexto.md`:
- Embedding `mpnet-base-v2` (mejor calidad que `MiniLM`, evita que todo caiga en un único tema gigante).
- Stopwords ES+EN en el vectorizador (si no, las etiquetas salen "de, que, la, el").
- HDBSCAN `cluster_selection_method='leaf'` + `min_samples=5` (temas específicos, no pocos temas muy anchos).
- `reduce_outliers` + `update_topics` (pasándole `vectorizer_model`/`ctfidf_model` explícitamente -- si no, `update_topics` resetea las stopwords).

In [ ]:
EMBEDDING_MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
MODEL_NAME = f'BERTopic+{EMBEDDING_MODEL_NAME} (modelo B: geo)'
MIN_TOPIC_SIZE = 15

SPANISH_STOPWORDS = {
    'de', 'la', 'que', 'el', 'en', 'y', 'a', 'los', 'del', 'se', 'las', 'por',
    'un', 'para', 'con', 'no', 'una', 'su', 'al', 'lo', 'como', 'mas', 'más',
    'pero', 'sus', 'le', 'ya', 'o', 'este', 'si', 'sí', 'porque', 'esta',
    'entre', 'cuando', 'muy', 'sin', 'sobre', 'tambien', 'también', 'me',
    'hasta', 'hay', 'donde', 'dónde', 'quien', 'quién', 'desde', 'todo',
    'nos', 'durante', 'todos', 'uno', 'les', 'ni', 'contra', 'otros', 'ese',
    'eso', 'ante', 'ellos', 'e', 'esto', 'mi', 'antes', 'algunos', 'que',
    'unos', 'yo', 'otro', 'otras', 'otra', 'el', 'tanto', 'esa', 'estos',
    'mucho', 'quienes', 'nada', 'muchos', 'cual', 'cuál', 'poco', 'ella',
    'estar', 'estas', 'algunas', 'algo', 'nosotros', 'mi', 'mis', 'tu', 'tú',
    'te', 'ti', 'tus', 'ellas', 'nosotras', 'vosotros', 'vosotras', 'os',
    'mio', 'mío', 'mia', 'mía', 'tuyo', 'tuya', 'suyo', 'suya', 'es', 'soy',
    'eres', 'somos', 'sois', 'son', 'esté', 'esta', 'estan', 'están', 'fue',
    'ser', 'voy', 'vamos', 'va', 'van', 'puede', 'pueden', 'hace', 'hacer',
    'gracias', 'hola', 'saludos', 'pues', 'asi', 'así', 'aqui', 'aquí',
    'alli', 'allí', 'ahi', 'ahí',
    # Especificas de reseñas de hotel, no aportan tema:
    'hotel', 'habitacion', 'habitación', 'noche', 'noches', 'estancia',
}

# Booking es muy multilingüe (a diferencia de YouTube/LosViajeros) -- sin esto,
# las reseñas en aleman/ruso/italiano/frances/portugues formaban sus propios
# "temas" cuya etiqueta eran solo palabras vacias de ese idioma sin filtrar
# (ej. aleman: "ein, wir, ist, auch, für, uns"), en vez de describir contenido.
GERMAN_STOPWORDS = {
    'der', 'die', 'das', 'und', 'ist', 'war', 'für', 'mit', 'auf', 'sich',
    'dem', 'den', 'des', 'ein', 'eine', 'einen', 'einem', 'einer', 'nicht',
    'auch', 'aber', 'wie', 'wir', 'ich', 'du', 'er', 'sie', 'es', 'was',
    'wenn', 'dass', 'so', 'noch', 'nur', 'sehr', 'hier', 'sind', 'hat',
    'haben', 'wird', 'kann', 'mehr', 'viel', 'gut', 'alle', 'aus', 'bei',
    'nach', 'vor', 'über', 'unter', 'zwischen', 'um', 'an', 'in', 'im',
    'zu', 'zum', 'zur', 'uns', 'euch', 'ihr', 'ihre', 'sein', 'seine',
    'diese', 'dieser', 'dieses', 'man', 'war', 'waren', 'wurde', 'werden',
}
ITALIAN_STOPWORDS = {
    'il', 'lo', 'la', 'i', 'gli', 'le', 'di', 'a', 'da', 'in', 'con', 'su',
    'per', 'tra', 'fra', 'un', 'uno', 'una', 'che', 'non', 'come', 'più',
    'anche', 'molto', 'questo', 'questa', 'questi', 'queste', 'sono', 'era',
    'è', 'ha', 'hanno', 'siamo', 'essere', 'avere', 'del', 'della', 'dei',
    'delle', 'al', 'alla', 'allo', 'nel', 'nella', 'dal', 'dalla', 'ma',
    'se', 'perché', 'quando', 'dove', 'chi', 'cosa', 'ci', 'si', 'tutto',
}
RUSSIAN_STOPWORDS = {
    'и', 'в', 'не', 'на', 'я', 'быть', 'он', 'с', 'что', 'а', 'по', 'это',
    'она', 'этот', 'к', 'но', 'они', 'мы', 'как', 'из', 'у', 'который',
    'то', 'за', 'свой', 'весь', 'год', 'от', 'так', 'о', 'для', 'ты',
    'же', 'все', 'тот', 'вы', 'если', 'уже', 'или', 'ни', 'бы', 'себя',
    'под', 'будет', 'был', 'была', 'было', 'были', 'есть', 'очень',
    'чтобы', 'при', 'без',
}
FRENCH_STOPWORDS = {
    'le', 'la', 'les', 'de', 'un', 'une', 'du', 'des', 'et', 'à', 'est',
    'il', 'elle', 'ils', 'elles', 'on', 'nous', 'vous', 'je', 'tu', 'que',
    'qui', 'pour', 'dans', 'avec', 'sur', 'par', 'ce', 'cette', 'ces',
    'se', 'son', 'sa', 'ses', 'ne', 'pas', 'plus', 'très', 'aussi', 'mais',
    'ou', 'où', 'comme', 'tout', 'tous', 'toutes', 'être', 'avoir', 'fait',
    'sont', 'était', 'été',
}
PORTUGUESE_STOPWORDS = {
    'o', 'a', 'os', 'as', 'de', 'do', 'da', 'dos', 'das', 'um', 'uma', 'e',
    'é', 'foi', 'ser', 'para', 'com', 'não', 'mais', 'muito', 'mas', 'ou',
    'quando', 'onde', 'que', 'se', 'este', 'esta', 'esses', 'essas', 'isso',
    'aqui', 'também', 'já', 'ainda', 'só', 'como', 'sua', 'seu', 'suas',
    'seus', 'nos', 'na', 'no', 'em', 'por',
}

ALL_STOPWORDS = (
    SPANISH_STOPWORDS | GERMAN_STOPWORDS | ITALIAN_STOPWORDS
    | RUSSIAN_STOPWORDS | FRENCH_STOPWORDS | PORTUGUESE_STOPWORDS
)

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

print(f'Cargando {EMBEDDING_MODEL_NAME!r} en GPU...')
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME, device='cuda' if torch.cuda.is_available() else 'cpu')

hdbscan_model = HDBSCAN(
    min_cluster_size=MIN_TOPIC_SIZE,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='leaf',
    prediction_data=True,
)
vectorizer_model = CountVectorizer(
    stop_words=list(ENGLISH_STOP_WORDS | ALL_STOPWORDS),
    ngram_range=(1, 2),
    min_df=2,
)
topic_model = BERTopic(
    embedding_model=embedding_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ClassTfidfTransformer(reduce_frequent_words=True),
    min_topic_size=MIN_TOPIC_SIZE,
    language='multilingual',
    calculate_probabilities=False,
    verbose=True,
)

In [ ]:
print(f'Entrenando BERTopic sobre {len(textos)} documentos (esto puede tardar varios minutos en GPU)...')
topics, probs = topic_model.fit_transform(textos)

was_outlier = [t == -1 for t in topics]
n_before = sum(was_outlier)
print(f'Reasignando outliers ({n_before} de {len(topics)})...')
topics = topic_model.reduce_outliers(textos, topics, strategy='c-tf-idf')
n_after = topics.count(-1)
print(f'Outliers: {n_before} -> {n_after}.')
probs = [None if wo else p for wo, p in zip(was_outlier, probs)]

# OJO: sin pasar vectorizer_model/ctfidf_model explicitamente aqui, update_topics
# resetea las stopwords y las etiquetas vuelven a salir "que, de, la, el".
topic_model.update_topics(
    textos,
    topics=topics,
    vectorizer_model=topic_model.vectorizer_model,
    ctfidf_model=topic_model.ctfidf_model,
)
print('Entrenamiento terminado.')

In [ ]:
topic_info = {}
for row in topic_model.get_topic_info().itertuples():
    words = [w for w, _ in topic_model.get_topic(row.Topic)] if row.Topic != -1 else []
    topic_info[row.Topic] = (', '.join(words[:6]) if words else 'outlier / sin tema claro', row.Count)

print('Topicos encontrados:')
for topic_id, (label, size) in sorted(topic_info.items(), key=lambda kv: -kv[1][1]):
    tag = 'outlier' if topic_id == -1 else f'#{topic_id}'
    print(f'  {tag:>8} ({size:>5} docs): {label}')

## Exportar resultados

Mismas columnas que la tabla `gold.nlp_topics` del proyecto, para que Claude pueda cargarlo directo con `analytics/topics/import_geo_results.py`.

In [ ]:
resultados = []
for (source, source_id, text), topic_id, prob in zip(zip(df['source'], df['source_id'], df['text']), topics, probs):
    label, size = topic_info.get(topic_id, (None, None))
    resultados.append({
        'source': source,
        'source_id': source_id,
        'text': text,
        'topic_id': int(topic_id),
        'topic_label': label,
        'topic_size': size,
        'probability': float(prob) if prob is not None else None,
        'model_name': MODEL_NAME,
    })

df_resultados = pd.DataFrame(resultados)
df_resultados.to_csv('geo_topics_results.csv', index=False)
print(f'{len(df_resultados)} filas guardadas en geo_topics_results.csv')

topic_model.save('bertopic_model_geo', serialization='safetensors', save_ctfidf=True, save_embedding_model=EMBEDDING_MODEL_NAME)
!zip -rq bertopic_model_geo.zip bertopic_model_geo
print('Modelo guardado en bertopic_model_geo.zip')

files.download('geo_topics_results.csv')
files.download('bertopic_model_geo.zip')